# E3DTI training on Kaggle

Before running anything, set these in the notebook sidebar under **Settings**:

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (or P100) |
| Internet | **On** — needed for `pip`, `git clone` and the Hugging Face data |
| Persistence | Files only |

If the GPU option is greyed out, the account needs phone verification:
Settings → Phone Verification. GPU quota is about 30 hours a week and a
session runs at most ~12 hours, so long runs **must** use `--resume`.

Two directories matter here:

- `/kaggle/temp` — scratch, deleted when the session ends. The repo goes here.
- `/kaggle/working` — saved when you click **Save Version**. Results go here.

Anything written anywhere else is lost when the session dies.

In [ ]:
# 1. Confirm a GPU is actually attached before spending quota on setup.
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then rerun.")

In [ ]:
# 2. Dependencies. torch, pandas, numpy and scikit-learn are already in the
#    Kaggle image; rdkit is not.
!pip install -q rdkit huggingface-hub

In [ ]:
# 3. Get the code. Clone into scratch, not into /kaggle/working, so the saved
#    output stays small.
BRANCH = "yogesh/drug-egnn"
REPO = "/kaggle/temp/final-project-9"

!rm -rf {REPO}
!git clone -q --branch {BRANCH} https://github.com/XingqiaoLin/final-project-9.git {REPO}
!cd {REPO} && git log --oneline -1

In [ ]:
# 4. Correctness gates. docs/experiment_protocol.md requires these to pass
#    before any training run counts.
!cd {REPO} && python -m unittest discover -s tests

In [ ]:
# 5. Smoke run: proves the pipeline works end to end and warms the Hugging
#    Face cache. Not a result. Takes a couple of minutes.
!cd {REPO} && python train_e3dti.py \
  --dataset biosnap --split random --seed 42 \
  --epochs 1 --batch 4 --dim 32 \
  --drug-layers 1 --protein-layers 1 \
  --max-train-pairs 32 --max-val-pairs 16 --max-test-pairs 16 \
  --output-root /kaggle/temp/runs-smoke

In [ ]:
# 6. One timed epoch at full model size, so the run budget is a measured
#    number rather than a guess. Multiply by --epochs to size the real run.
import time

start = time.time()
!cd {REPO} && python train_e3dti.py \
  --dataset biosnap --split random --seed 1 \
  --epochs 1 --batch 32 --dim 128 \
  --drug-layers 4 --protein-layers 4 \
  --output-root /kaggle/temp/runs-timing
print(f"\none full epoch: {time.time() - start:.0f}s")

In [ ]:
# 7. The real run. --resume writes last.pt every epoch and restores model,
#    optimizer, scheduler, history, early-stopping counters and RNG state.
#    If the session dies, rerun this exact cell in a new session and it picks
#    up at the next epoch. If results.json already exists it exits at once,
#    so rerunning is always safe.
!cd {REPO} && python train_e3dti.py \
  --dataset biosnap --split random --seed 1 \
  --output-root /kaggle/working/runs \
  --resume

In [ ]:
# 8. What survived, and how far each run got.
import json
from pathlib import Path

for results in sorted(Path("/kaggle/working/runs").rglob("results.json")):
    data = json.loads(results.read_text())
    config = data["config"]
    print(
        f"{config['dataset']}/{config['split']} {config['ablation']} "
        f"seed={config['seed']} epoch={data['best_epoch']} "
        f"AUROC={data['test']['auroc']:.4f} AUPRC={data['test']['auprc']:.4f}"
    )

for checkpoint in sorted(Path("/kaggle/working/runs").rglob("last.pt")):
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    print(f"{checkpoint.parent}: in progress, reached epoch {state['epoch']}")

## When a session dies

1. Click **Save Version** → *Save & Run All (Commit)* so `/kaggle/working`
   persists. A session that times out without a commit loses everything.
2. In the new session, add the previous version's output as a data source so
   `/kaggle/working/runs` carries over, then rerun cells 1–3 and cell 7.
3. Cell 7 reads `last.pt` and continues from the next epoch.

Pass the same `--epochs` on every resume. The cosine schedule is derived from
it, so changing it mid-run changes the learning rate for the epochs that
remain.

## Budgeting the matrix

Cell 6 gives seconds per epoch. With `--epochs 50` and `--patience 10`, a run
costs somewhere between 10 and 50 epochs of that. At roughly 30 GPU-hours a
week, decide what fits **before** launching: the full matrix in
`docs/experiment_protocol.md` is 3 datasets x 2 splits x 3 seeds x 4 ablations
= 72 runs, plus the two DrugBAN arms.